In [40]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [41]:
df = pd.read_csv("../data/adult.csv")
df_cleaned = df.replace('?', np.nan)
df_cleaned.dropna(inplace=True)
inconsistent_mask = (
    (df_cleaned['salary K$'] <= 50) & (df_cleaned['salary'] != '<=50K')
) | (
    (df_cleaned['salary K$'] > 50) & (df_cleaned['salary'] != '>50K')
)
df_cleaned.loc[inconsistent_mask, 'salary'] = np.where(
    df_cleaned.loc[inconsistent_mask, 'salary K$'] <= 50,
    '<=50K',
    '>50K'
)
df.head()

,Unnamed: 0,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
0,0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,39
1,1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,35
2,2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,27
3,3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,43
4,4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,25


# Task 1
Print the count of men and women in the dataset.

In [42]:
sex_counts = df_cleaned['sex'].value_counts()
print("Count of men and women in the dataset:")
print(sex_counts)

Count of men and women in the dataset:
sex
Male      20380
Female     9782
Name: count, dtype: int64


# Task 2
Find the average age of men in dataset

In [43]:
men_df = df_cleaned[df_cleaned['sex'] == 'Male']
average_age_men = men_df['age'].mean()
print(f"The average age of men in the dataset is: {average_age_men:.2f}")

The average age of men in the dataset is: 39.18


# Task 3
Get the percentage of people from Poland (native-country)

In [44]:
total_people = len(df_cleaned)
people_from_poland = len(df_cleaned[df_cleaned['native-country'] == 'Poland'])
if total_people > 0:
    percentage_from_poland = (people_from_poland / total_people) * 100
    print(f"The percentage of people from Poland in the dataset is: {percentage_from_poland:.2f}%")
else:
    print("The dataset is empty after cleaning.")

The percentage of people from Poland in the dataset is: 0.19%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [45]:
grouped_by_salary = df_cleaned.groupby('salary')['age']
age_stats = grouped_by_salary.agg(['mean', 'std'])
print("Mean and Standard Deviation of Age by Salary Group:")
print(age_stats)

Mean and Standard Deviation of Age by Salary Group:
            mean        std
salary                     
<=50K   36.60806  13.464631
>50K    43.95911  10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [46]:
higher_education_degrees = [
    'Bachelors',
    'Prof-school',
    'Assoc-acdm',
    'Assoc-voc',
    'Masters',
    'Doctorate'
]

condition_mask = (df_cleaned['salary'] == '>50K') & (~df_cleaned['education'].isin(higher_education_degrees))
people_without_higher_education = df_cleaned[condition_mask]
if not people_without_higher_education.empty:
    print("Yes, there are people without a higher education who earn >50K.")
    print(f"Count: {len(people_without_higher_education)}")
    print("\nSample of these people:")
    print(people_without_higher_education.head())
else:
    print("No, there are no people in the dataset without a higher education who earn >50K.")

Yes, there are people without a higher education who earn >50K.
Count: 3178

Sample of these people:
    Unnamed: 0  age         workclass     education      marital-status  \
7            7   52  Self-emp-not-inc       HS-grad  Married-civ-spouse   
10          10   37           Private  Some-college  Married-civ-spouse   
55          55   43           Private  Some-college  Married-civ-spouse   
67          67   53           Private       HS-grad  Married-civ-spouse   
68          68   49      Self-emp-inc  Some-college  Married-civ-spouse   

         occupation relationship   race     sex  hours-per-week  \
7   Exec-managerial      Husband  White    Male              45   
10  Exec-managerial      Husband  Black    Male              80   
55     Tech-support      Husband  White    Male              40   
67     Adm-clerical         Wife  White  Female              40   
68  Exec-managerial      Husband  White    Male              50   

   native-country salary  salary K$  
7   Uni

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [47]:
age_stats_by_education = df_cleaned.groupby('education')['age'].describe()
print("Descriptive Statistics of Age for Each Education Type:")
print(age_stats_by_education)

Descriptive Statistics of Age for Each Education Type:
               count       mean        std   min   25%   50%   75%   max
education                                                               
10th           820.0  37.897561  16.225795  17.0  23.0  36.0  52.0  90.0
11th          1048.0  32.363550  15.089307  17.0  18.0  28.5  43.0  90.0
12th           377.0  32.013263  14.373710  17.0  19.0  28.0  41.0  79.0
1st-4th        151.0  44.622517  14.929051  19.0  33.0  44.0  56.0  81.0
5th-6th        288.0  41.649306  14.754622  17.0  28.0  41.0  53.0  82.0
7th-8th        557.0  47.631957  15.737479  17.0  34.0  49.0  60.0  90.0
9th            455.0  40.303297  15.335754  17.0  28.0  38.0  53.0  90.0
Assoc-acdm    1008.0  37.286706  10.509755  19.0  29.0  36.0  44.0  90.0
Assoc-voc     1307.0  38.246366  11.181253  19.0  30.0  37.0  45.0  84.0
Bachelors     5044.0  38.641554  11.577566  19.0  29.0  37.0  46.0  90.0
Doctorate      375.0  47.130667  11.471727  24.0  39.0  47.0  54.0  8

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [48]:
men_df = df_cleaned[df_cleaned['sex'] == 'Male'].copy()
married_mask = men_df['marital-status'].str.startswith('Married')
non_married_mask = ~married_mask
married_men_salary_counts = men_df[married_mask]['salary'].value_counts(normalize=True) * 100
non_married_men_salary_counts = men_df[non_married_mask]['salary'].value_counts(normalize=True) * 100
print("Percentage of Married Men by Salary:")
print(married_men_salary_counts)
print("\nPercentage of Non-Married Men by Salary:")
print(non_married_men_salary_counts)
married_high_earners = married_men_salary_counts.get('>50K', 0)
non_married_high_earners = non_married_men_salary_counts.get('>50K', 0)
if married_high_earners > non_married_high_earners:
    print("Based on the data, married men have a higher percentage of earners with a salary >50K.")
elif married_high_earners < non_married_high_earners:
    print("Based on the data, non-married men have a higher percentage of earners with a salary >50K.")
else:
    print("The percentage of earners with a salary >50K is the same for both married and non-married men.")

Percentage of Married Men by Salary:
salary
<=50K    55.201566
>50K     44.798434
Name: proportion, dtype: float64

Percentage of Non-Married Men by Salary:
salary
<=50K    91.150559
>50K      8.849441
Name: proportion, dtype: float64
Based on the data, married men have a higher percentage of earners with a salary >50K.


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [49]:
max_hours = df_cleaned['hours-per-week'].max()
people_with_max_hours = df_cleaned['hours-per-week'].value_counts().get(max_hours, 0)
print(f"The maximum hours worked per week is: {max_hours}")
print(f"The number of people who work the same amount of hours per week is: {people_with_max_hours}")

The maximum hours worked per week is: 99
The number of people who work the same amount of hours per week is: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight their connection.

In [50]:
correlation_matrix = df_cleaned.corr(numeric_only=True)
correlation_matrix

,Unnamed: 0,age,hours-per-week,salary K$
Unnamed: 0,1.000000,-0.001126,-0.001890,0.000129
age,-0.001126,1.000000,0.101599,0.208203
hours-per-week,-0.001890,0.101599,1.000000,0.196378
salary K$,0.000129,0.208203,0.196378,1.000000
